In [1]:
# Import Required Libs...

import pandas as pd
import requests
import json

In [2]:
# 1. Get the dataset from Localfile

# Documents
with open("documents-with-ids.json", "rt") as f_in:
    documents = json.load(f_in)

# Ground-truth
df_ground_truth = pd.read_csv("ground-truth-data.csv")
ground_truth = df_ground_truth.to_dict(orient='records')

In [3]:
from tqdm.auto import tqdm

def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        #print(f"Query dict: {q}")
        doc_id = q['Document']
        results = search_function(q)
        #print(f"Search results: {results}")
        #print("---")
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Question: 1 ...

import minsearch

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course", "id"]
)

index.fit(documents)

In [5]:
def minsearch_search(query, course):
    boost = {'question': 1.5, 'section': 0.1}

    results = index.search(
        query=query,
        filter_dict={'course': course},
        boost_dict=boost,
        num_results=5
    )

    return results

In [6]:
evaluate(ground_truth, lambda q: minsearch_search(q['Question'], q['Course']))

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4735/4735 [00:15<00:00, 308.55it/s]


{'hit_rate': 0.8500527983104541, 'mrr': 0.722509679690251}

In [7]:
# Question: 2 ...

from minsearch import VectorSearch

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

In [9]:
texts = []

for doc in documents:
    t = doc['question']
    texts.append(t)

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X = pipeline.fit_transform(texts)

In [10]:
vindex = VectorSearch(keyword_fields={'course'})
vindex.fit(X, documents)

In [11]:
def vector_search(query):
    query_vector = pipeline.transform([query])[0]
    print(f"Vectorizing and searching for query: '{query}'")
    return vindex.search(query_vector)

In [78]:
result = evaluate(ground_truth, lambda q: vector_search(q['Question']))

  3%|███▌                                                                                                                    | 140/4735 [00:00<00:06, 694.00it/s]

Vectorizing and searching for query: 'When does the course officially begin and how can I join the live session?'
Vectorizing and searching for query: 'What is the exact date and time for the first Office Hours session?'
Vectorizing and searching for query: 'How do I subscribe to the course schedule and where can I find it?'
Vectorizing and searching for query: 'What steps are required to register for the course before it starts?'
Vectorizing and searching for query: 'Which platforms should I join to receive announcements and interact with other students?'
Vectorizing and searching for query: 'What do I need to know before starting this course?'
Vectorizing and searching for query: 'Are there any requirements I should meet to take this course?'
Vectorizing and searching for query: 'Can you tell me about the prerequisites for enrolling in this course?'
Vectorizing and searching for query: 'What skills or knowledge should I have before joining this course?'
Vectorizing and searching for 

  4%|█████▎                                                                                                                  | 210/4735 [00:00<00:06, 694.59it/s]

Vectorizing and searching for query: 'Can I use AWS for this course instead of the recommended platform?'
Vectorizing and searching for query: 'What should I do if I choose AWS for the course?'
Vectorizing and searching for query: 'Will the final capstone project be different if I use AWS?'
Vectorizing and searching for query: 'How can I get help if I encounter issues with AWS during the course?'
Vectorizing and searching for query: 'Are there many other students using AWS in this course?'
Vectorizing and searching for query: 'Are there any live Zoom calls besides the Office Hour?'
Vectorizing and searching for query: 'Will there be any additional live sessions during the course?'
Vectorizing and searching for query: 'How often are live Zoom calls scheduled outside of Office Hours?'
Vectorizing and searching for query: 'Can I expect any unannounced live Zoom sessions during the course?'
Vectorizing and searching for query: 'What are the chances of having extra live Zoom meetings beyond

  7%|████████▉                                                                                                               | 352/4735 [00:00<00:06, 654.30it/s]

...

Vectorizing and searching for query: 'How can I fix the issue where MLflow CLI doesn't display experiments?'
Vectorizing and searching for query: 'What environment variable needs to be set for MLflow CLI to work properly?'
Vectorizing and searching for query: 'Why is the MLflow CLI not returning any experiments, and how do I resolve it?'
Vectorizing and searching for query: 'How can I make MLflow CLI commands recognize experiments run with the tracking server?'
Vectorizing and searching for query: 'Why do I need to set the MLFLOW_TRACKING_URI environment variable for MLflow CLI?'
Vectorizing and searching for query: 'What is the correct format for the MLFLOW_TRACKING_URI when using SQLite?'
Vectorizing and searching for query: 'Why does the 'mlflow gc' command not automatically use the tracking URI?'
Vectorizing and searching for query: 'How can I explicitly pass the tracking URI to MLflow CLI commands that don't recognize it automatically?'
Vectorizing and searching for query: 'How ca

 97%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉   | 4614/4735 [00:07<00:00, 631.75it/s]

Vectorizing and searching for query: 'Which command should I use to build a Docker container from a Dockerfile?'
Vectorizing and searching for query: 'What does the --rm flag do when running a Docker container?'
Vectorizing and searching for query: 'Is the name 'mlops-learn' in the example commands a required naming convention for Docker images?'
Vectorizing and searching for query: 'How can I run both Flask with Gunicorn and MLFlow server in the same Docker container?'
Vectorizing and searching for query: 'Why does defining both services in the Dockerfile's CMD only run MLFlow and not Flask?'
Vectorizing and searching for query: 'What is the recommended approach to run multiple services in a single Docker container?'
Vectorizing and searching for query: 'How do I create a wrapper script to start multiple services in a Docker container?'
Vectorizing and searching for query: 'What permissions should I set for the shell scripts used to run multiple services in a Docker container?'
Vector

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4735/4735 [00:07<00:00, 634.04it/s]

Vectorizing and searching for query: 'What should I do if I encounter a 'IllegalLocationConstraintException' while using the boto3 client to create a bucket?'
Vectorizing and searching for query: 'Why does the error 'The unspecified location constraint is incompatible for the region specific endpoint' occur during bucket creation?'
Vectorizing and searching for query: 'What is the correct way to specify a location constraint when creating an S3 bucket with boto3 in localstack?'
Vectorizing and searching for query: 'How can I modify my bucket creation code to include the required location constraint for localstack compatibility?'
Vectorizing and searching for query: 'Why do I see an error with a long object reference when running AWS CLI commands?'
Vectorizing and searching for query: 'How can I resolve the AWSRequest object error in AWS CLI?'
Vectorizing and searching for query: 'What should I do if AWS CLI shows a botocore.awsrequest.AWSRequest error?'
Vectorizing and searching for qu

In [85]:
print(f"MRR result is: {result['mrr']:.2f}")

MRR result is: 0.62


In [13]:
# Question: 3...

texts = []

for doc in documents:
    t = doc['question'] + ' ' + doc['text']
    texts.append(t)

In [14]:
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X = pipeline.fit_transform(texts)

In [15]:
vindex = VectorSearch(keyword_fields={'course'})
vindex.fit(X, documents)

In [16]:
def vector_search_function_q3(q):
    query_text = q['Question']
    query_vector = pipeline.transform([query_text])[0]
    results = vindex.search(
        query_vector,
        filter_dict={'course': q['Course']},
    )
    return results

In [17]:
results_q3 = evaluate(ground_truth, lambda q: vector_search_function_q3(q))
hit_rate_q3 = results_q3['hit_rate']
print(f"Hit Rate for vector search on 'question' + 'text' field (Q3): {hit_rate_q3}")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4735/4735 [00:07<00:00, 601.36it/s]

Hit Rate for vector search on 'question' + 'text' field (Q3): 0.8984160506863781


In [56]:
# Question: 4...
from qdrant_client import QdrantClient, models
from tqdm.auto import tqdm

# --- 0. Configuration ---
model_handle = "jinaai/jina-embeddings-v2-small-en"
limit = 5 # As specified in Q4
collection_name = "llm_zoomcamp_embeddings"
embedding_dimension = 512

In [23]:
#!pip install jina sentence-transformers qdrant-client[fastembed]

In [24]:
!pip show qdrant-client
!pip show fastembed

Name: qdrant-client
Version: 1.15.0
Summary: Client library for the Qdrant vector search engine
Home-page: https://github.com/qdrant/qdrant-client
Author: Andrey Vasnetsov
Author-email: andrey@qdrant.tech
License: Apache-2.0
Location: /usr/local/python/3.12.1/lib/python3.12/site-packages
Requires: grpcio, httpx, numpy, portalocker, protobuf, pydantic, urllib3
Required-by: 
Name: fastembed
Version: 0.7.1
Summary: Fast, light, accurate library built for retrieval embedding generation
Home-page: https://github.com/qdrant/fastembed
Author: Qdrant Team
Author-email: info@qdrant.tech
License: Apache License
Location: /usr/local/python/3.12.1/lib/python3.12/site-packages
Requires: huggingface-hub, loguru, mmh3, numpy, onnxruntime, pillow, py-rust-stemmers, requests, tokenizers, tqdm
Required-by: 


In [57]:
from sentence_transformers import SentenceTransformer

print(f"Loading SentenceTransformer model: {model_handle}...")
embedding_model = SentenceTransformer(model_handle)
loaded_model_dimension = embedding_model.get_sentence_embedding_dimension()
print(f"DEBUG: Dimension of the model loaded by SentenceTransformer: {loaded_model_dimension}")

#if loaded_model_dimension != 768:
    #raise ValueError(f"CRITICAL ERROR: Loaded model dimension ({loaded_model_dimension}) does not match expected Jina dimension (768).")

Loading SentenceTransformer model: jinaai/jina-embeddings-v2-small-en...


Some weights of BertModel were not initialized from the model checkpoint at jinaai/jina-embeddings-v2-small-en and are newly initialized: ['embeddings.position_embeddings.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0.output.dense.weight', 'encoder.layer.1.intermediate.dense.bias', 'encoder.layer.1.intermediate.dense.weight', 'encoder.layer.1.output.LayerNorm.bias', 'encoder.layer.1.output.LayerNorm.weight', 'encoder.layer.1.output.dense.bias', 'encoder.layer.1.output.dense.weight', 'encoder.layer.2.intermediate.dense.bias', 'encoder.layer.2.intermediate.dense.weight', 'encoder.layer.2.output.LayerNorm.bias', 'encoder.layer.2.output.LayerNorm.weight', 'encoder.layer.2.output.dense.bias', 'encoder.layer.2.output.dense.weight', 'encoder.layer.3.intermediate.dense.bias', 'encoder.layer.3.intermediate.den

DEBUG: Dimension of the model loaded by SentenceTransformer: 512


In [49]:
original_hash_id_to_qdrant_int_id = {}
processed_documents = [] # A new list if we modify documents
for i, doc in enumerate(documents):
    # Assume doc['id'] holds your original hash string, e.g., 'c02e79ef'
    original_id = doc['id']
    new_int_id = i # Use the sequential index as the Qdrant ID

    original_hash_id_to_qdrant_int_id[original_id] = new_int_id
    
    # Optionally, you can modify the doc in place or create a new one with the int ID
    # For payload, it's fine to keep the original hash string ID
    doc_copy = doc.copy()
    doc_copy['qdrant_id'] = new_int_id # Store the new int ID in payload for clarity if needed
    processed_documents.append(doc_copy)

print(f"DEBUG: Created ID mapping for {len(original_hash_id_to_qdrant_int_id)} documents.")

DEBUG: Created ID mapping for 947 documents.


In [51]:
points_for_qdrant = []
print("Generating embeddings for documents and preparing for upsert...")
for i, doc in enumerate(processed_documents): # Use processed_documents here if you created a new list
    combined_text = doc['question'] + ' ' + doc['text']
    embedding = embedding_model.encode(combined_text).tolist()
    
    # Use the new integer ID for the PointStruct
    qdrant_point_id = original_hash_id_to_qdrant_int_id[doc['id']] # Get the int ID based on original hash

    points_for_qdrant.append(
        models.PointStruct(
            id=qdrant_point_id, # THIS IS THE CRUCIAL CHANGE for the Point ID
            vector=embedding,
            payload=doc # Store the original document (with its hash ID) as payload
        )
    )

print(f"DEBUG: Number of points prepared for Qdrant: {len(points_for_qdrant)}")

if not points_for_qdrant:
    raise ValueError("No points were prepared for Qdrant. Check your document loading and text processing.")

Generating embeddings for documents and preparing for upsert...
DEBUG: Number of points prepared for Qdrant: 948


In [58]:
client = QdrantClient(":memory:")

In [59]:
# Recreate Collections
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=embedding_dimension, distance=models.Distance.COSINE),
)

/tmp/ipykernel_46481/1637442889.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [60]:
print(f"Upserting {len(points_for_qdrant)} documents into Qdrant...")
client.upsert(
    collection_name=collection_name,
    wait=True,
    points=points_for_qdrant
)
print(f"Upserted {len(points_for_qdrant)} documents into Qdrant collection '{collection_name}' successfully.")

Upserting 948 documents into Qdrant...
Upserted 948 documents into Qdrant collection 'llm_zoomcamp_embeddings' successfully.


In [61]:
# --- 3. Define the Qdrant Search Function (FIXED: Using client.search) ---
def qdrant_search_with_st(q_entry, limit=limit, collection_name=collection_name, embedding_model_st=embedding_model, qdrant_client=client):
    """
    Performs a vector search in Qdrant for a given query dictionary,
    using a pre-initialized SentenceTransformer model for query embedding.
    """
    query_text = q_entry['Question']

    # Generate embedding for the query using the SentenceTransformer model
    query_embedding = embedding_model_st.encode(query_text).tolist()

    # Perform the search in Qdrant using client.search()
    search_results = qdrant_client.search( # <-- CHANGED FROM client.query() to client.search()
        collection_name=collection_name,
        query_vector=query_embedding, # This parameter name is correct for search()
        limit=limit,
        with_payload=True # Crucial to get the original document info back
    )

    # Format results to match what `evaluate` expects (list of dicts with 'id')
    formatted_results = []
    for result in search_results:
        # The payload should contain your original document, including its 'id'
        # Ensure your original documents stored in payload have an 'id' key
        formatted_results.append({
            'id': result.payload.get('id'),
            **result.payload # Include all other payload data if needed by other parts of the evaluation
        })
    return formatted_results

In [62]:
print("\nStarting evaluation...")
results = evaluate(ground_truth, qdrant_search_with_st)


Starting evaluation...


  0%|                                                                                                                                   | 0/4735 [00:00<?, ?it/s]/tmp/ipykernel_46481/2774459501.py:13: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = qdrant_client.search( # <-- CHANGED FROM client.query() to client.search()
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4735/4735 [01:43<00:00, 45.63it/s]


In [63]:
print("\n--- Evaluation Results (Qdrant with SentenceTransformer) ---")
print(f"Hit Rate: {results['hit_rate']:.2f}")
print(f"MRR: {results['mrr']:.2f}")


--- Evaluation Results (Qdrant with SentenceTransformer) ---
Hit Rate: 0.07
MRR: 0.05


In [64]:
# Question 5 ...

In [72]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import numpy as np

In [67]:
# Define the cosine similarity function
def cosine(u, v):
    u_norm = np.sqrt(u.dot(u))
    v_norm = np.sqrt(v.dot(v))
    # Handle cases where norm might be zero to avoid division by zero
    if u_norm == 0 or v_norm == 0:
        return 0.0
    return u.dot(v) / (u_norm * v_norm)

In [70]:
# 1. Load the results CSV
df_results = pd.read_csv('results-gpt4o-mini.csv')

print(f"Loaded {len(df_results)} rows from {results_url}")

Loaded 1830 rows from https://github.com/DataTalksClub/llm-zoomcamp/blob/main/03-evaluation/rag_evaluation/data/results-gpt4o-mini.csv


In [73]:
# 4. Compute cosine similarity for each pair
cosine_similarities = []

for index, row in df_results.iterrows():
    v_llm = pipeline.transform([row.answer_llm]).flatten()
    v_orig = pipeline.transform([row.answer_orig]).flatten()
    
    # Compute cosine between them
    sim = cosine(v_llm, v_orig)
    cosine_similarities.append(sim)

# 5. Calculate the average cosine similarity
average_cosine_similarity = np.mean(cosine_similarities)

print(f"\nAverage Cosine Similarity: {average_cosine_similarity:.2f}")


Average Cosine Similarity: 0.75


In [74]:
!pip install rouge

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [75]:
from rouge import Rouge

rouge_scorer = Rouge()

rouge_1_f1_scores = []

print(f"\nComputing ROUGE scores for {len(df_results)} pairs...")


Computing ROUGE scores for 1830 pairs...


In [76]:
for index, row in df_results.iterrows():
    llm_answer = str(row.answer_llm)  # Convert to string to handle potential NaNs
    original_answer = str(row.answer_orig) # Convert to string

    # get_scores might throw an error if one of the inputs is empty/nan.
    # We'll use a try-except block or filter out empty strings.
    if llm_answer.strip() == "" or original_answer.strip() == "":
        # print(f"Skipping row {index} due to empty answer.")
        continue

    try:
        scores = rouge_scorer.get_scores(llm_answer, original_answer)[0]
        rouge_1_f1_scores.append(scores['rouge-1']['f'])
    except ValueError as e:
        # Handle cases where Rouge might fail (e.g., extremely short texts)
        # print(f"Could not compute ROUGE for row {index}: {e}")
        continue


average_rouge_1_f1 = np.mean(rouge_1_f1_scores)

print(f"Average ROUGE-1 F1 Score: {average_rouge_1_f1:.2f}")

Average ROUGE-1 F1 Score: 0.35
